# Train YOLO26s for exam cheating-object detection on Kaggle

Dataset: [ahmedezzat02/datazeft](https://www.kaggle.com/datasets/ahmedezzat02/datazeft/data)

This notebook trains a COCO-pretrained **YOLO26s** detector on the 7-class Illegal-Tools dataset. YOLO26s is selected instead of YOLOv8 because its updated training recipe includes small-target-aware label assignment, while the `s` scale remains practical on a Kaggle T4/P100 GPU. The output is a regular Ultralytics `.pt` checkpoint.

Before running:

1. Create a Kaggle Notebook and upload this `.ipynb`.
2. Click **Add Input** and attach `ahmedezzat02/datazeft`.
3. In **Notebook options**, select a GPU accelerator (T4 x2, T4, or P100).
4. Enable Internet so Ultralytics and `yolo26s.pt` can be installed/downloaded.
5. Run all cells from top to bottom.

Class IDs are kept unchanged, but names are normalized for this project:

| Dataset ID | Original | Project name |
|---:|---|---|
| 0 | Book | cheat_sheet |
| 1 | Earphone | earphone |
| 2 | Mobile_phone | smartphone |
| 3 | cap | cap |
| 4 | headset | headset |
| 5 | smart_watch | smartwatch |
| 6 | sunglasses | sunglasses |


In [ ]:
# Install a YOLO26-capable Ultralytics release.
%pip install -q -U "ultralytics>=8.4.0,<9" pyyaml seaborn


In [ ]:
from __future__ import annotations

import json
import math
import os
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from IPython.display import FileLink, display
from PIL import Image
from ultralytics import YOLO, __version__ as ultralytics_version

SEED = 42
random.seed(SEED)

print("Ultralytics:", ultralytics_version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Enable a GPU accelerator in Kaggle Notebook options before training.")


## 1. Locate and audit the attached dataset

The Kaggle dataset normally appears at `/kaggle/input/datazeft`. The resolver below also handles a changed mount-folder name.

This dataset contains both 5-column YOLO detection boxes and YOLO segmentation polygons. The audit accepts both formats, measures each polygon using its enclosing box, and writes a detection-only label copy under `/kaggle/working` before training. The read-only Kaggle input remains unchanged.

In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/illegal_tools_yolo26")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def find_dataset_root() -> Path:
    preferred = KAGGLE_INPUT / "datazeft"
    if (preferred / "train" / "images").is_dir():
        return preferred
    candidates = []
    for train_images in KAGGLE_INPUT.rglob("train/images"):
        root = train_images.parent.parent
        if (root / "valid" / "images").is_dir() and (root / "test" / "images").is_dir():
            candidates.append(root)
    if not candidates:
        raise FileNotFoundError(
            "Dataset not found. Use Add Input and attach ahmedezzat02/datazeft."
        )
    return sorted(candidates, key=lambda path: len(str(path)))[0]

DATASET_ROOT = find_dataset_root()
print("Dataset root:", DATASET_ROOT)
print("Top-level files:", sorted(path.name for path in DATASET_ROOT.iterdir())[:20])


In [ ]:
ORIGINAL_NAMES = [
    "Book",
    "Earphone",
    "Mobile_phone",
    "cap",
    "headset",
    "smart_watch",
    "sunglasses",
]
PROJECT_NAMES = [
    "cheat_sheet",
    "earphone",
    "smartphone",
    "cap",
    "headset",
    "smartwatch",
    "sunglasses",
]

source_yaml_path = DATASET_ROOT / "data.yaml"
if source_yaml_path.is_file():
    source_yaml = yaml.safe_load(source_yaml_path.read_text(encoding="utf-8"))
    print("Original data.yaml names:", source_yaml.get("names"))
    assert int(source_yaml.get("nc", len(source_yaml.get("names", [])))) == 7

fixed_yaml = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {index: name for index, name in enumerate(PROJECT_NAMES)},
}
DATA_YAML = WORK_ROOT / "illegal_tools_project.yaml"
DATA_YAML.write_text(yaml.safe_dump(fixed_yaml, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))


In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_DIRS = {"train": "train", "val": "valid", "test": "test"}

def list_images(directory: Path) -> list[Path]:
    return sorted(path for path in directory.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES)

def source_group(path: Path) -> str:
    # Roboflow names augmented copies as original_name.rf.<hash>.jpg.
    return path.name.split(".rf.", 1)[0]

def parse_yolo_annotation(raw_line: str) -> tuple[int, float, float, float, float, str]:
    """Return class_id, xywh box, and source format for a box or polygon row."""
    parts = raw_line.split()
    if len(parts) < 5:
        raise ValueError(f"expected at least 5 columns, found {len(parts)}")

    class_value = float(parts[0])
    if not math.isfinite(class_value) or not class_value.is_integer():
        raise ValueError(f"class ID must be an integer, found {parts[0]!r}")
    class_id = int(class_value)
    if not 0 <= class_id < len(PROJECT_NAMES):
        raise ValueError(f"class ID {class_id} is outside 0..{len(PROJECT_NAMES) - 1}")

    coordinates = [float(value) for value in parts[1:]]
    if not all(math.isfinite(value) and 0 <= value <= 1 for value in coordinates):
        raise ValueError("coordinates must be finite and normalized to 0..1")

    if len(coordinates) == 4:
        x, y, width, height = coordinates
        annotation_format = "bbox"
    elif len(coordinates) >= 6 and len(coordinates) % 2 == 0:
        xs = coordinates[0::2]
        ys = coordinates[1::2]
        x_min, x_max = min(xs), max(xs)
        y_min, y_max = min(ys), max(ys)
        width, height = x_max - x_min, y_max - y_min
        x, y = (x_min + x_max) / 2, (y_min + y_max) / 2
        annotation_format = "polygon"
    else:
        raise ValueError(
            "expected xywh or at least three polygon points; "
            f"found {len(coordinates)} coordinate values"
        )

    if width <= 0 or height <= 0:
        raise ValueError("annotation has zero width or height")
    return class_id, x, y, width, height, annotation_format

split_images = {}
class_counts = Counter()
annotation_format_counts = Counter()
box_areas = defaultdict(list)
invalid_rows = []
missing_labels = []
groups_by_split = defaultdict(set)

for split, folder in SPLIT_DIRS.items():
    image_dir = DATASET_ROOT / folder / "images"
    label_dir = DATASET_ROOT / folder / "labels"
    images = list_images(image_dir)
    split_images[split] = images
    for image_path in images:
        groups_by_split[source_group(image_path)].add(split)
        label_path = label_dir / f"{image_path.stem}.txt"
        if not label_path.is_file():
            missing_labels.append(str(label_path))
            continue
        for line_number, raw_line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
            if not raw_line.strip():
                continue
            try:
                class_id, x, y, width, height, annotation_format = parse_yolo_annotation(raw_line)
            except (ValueError, IndexError) as error:
                invalid_rows.append((str(label_path), line_number, str(error), raw_line[:240]))
                continue
            class_counts[class_id] += 1
            annotation_format_counts[annotation_format] += 1
            box_areas[class_id].append(width * height)

leaked_groups = {group: splits for group, splits in groups_by_split.items() if len(splits) > 1}
audit_rows = []
for split, images in split_images.items():
    audit_rows.append({"split": split, "images": len(images)})
display(pd.DataFrame(audit_rows))

class_rows = []
for class_id, name in enumerate(PROJECT_NAMES):
    areas = box_areas[class_id]
    class_rows.append({
        "id": class_id,
        "class": name,
        "boxes": class_counts[class_id],
        "median_box_area_%": round(100 * (pd.Series(areas).median() if areas else 0), 3),
        "small_boxes_<1%": sum(area < 0.01 for area in areas),
    })
display(pd.DataFrame(class_rows))

print("Missing label files:", len(missing_labels))
print("Invalid annotation rows:", len(invalid_rows))
print("Annotation formats:", dict(annotation_format_counts))
print("Source groups present in more than one split:", len(leaked_groups))
if invalid_rows:
    print("First invalid rows:", invalid_rows[:10])
if leaked_groups:
    print("WARNING: augmented variants of the same source may cross splits; metrics can be optimistic.")
    print("First cross-split groups:", list(leaked_groups.items())[:10])

assert split_images["train"], "Training images were not found"
assert split_images["val"], "Validation images were not found"
assert split_images["test"], "Test images were not found"
assert not invalid_rows, "Fix truly invalid YOLO rows before training"

# Normalize every annotation to a 5-column detection box. This avoids mixed
# bbox/polygon behavior in the detection dataloader while keeping images in
# the read-only Kaggle input through lightweight directory symlinks.
DETECTION_DATASET_ROOT = WORK_ROOT / "detection_dataset"
for split, folder in SPLIT_DIRS.items():
    source_image_dir = DATASET_ROOT / folder / "images"
    source_label_dir = DATASET_ROOT / folder / "labels"
    target_split_dir = DETECTION_DATASET_ROOT / folder
    target_image_dir = target_split_dir / "images"
    target_label_dir = target_split_dir / "labels"
    target_split_dir.mkdir(parents=True, exist_ok=True)
    target_label_dir.mkdir(parents=True, exist_ok=True)
    if not target_image_dir.exists():
        target_image_dir.symlink_to(source_image_dir, target_is_directory=True)

    for image_path in split_images[split]:
        source_label_path = source_label_dir / f"{image_path.stem}.txt"
        normalized_lines = []
        for raw_line in source_label_path.read_text(encoding="utf-8").splitlines():
            if not raw_line.strip():
                continue
            class_id, x, y, width, height, _ = parse_yolo_annotation(raw_line)
            normalized_lines.append(
                f"{class_id} {x:.10g} {y:.10g} {width:.10g} {height:.10g}"
            )
        normalized_text = "\n".join(normalized_lines)
        if normalized_text:
            normalized_text += "\n"
        (target_label_dir / f"{image_path.stem}.txt").write_text(
            normalized_text,
            encoding="utf-8",
        )

fixed_yaml["path"] = str(DETECTION_DATASET_ROOT)
DATA_YAML.write_text(yaml.safe_dump(fixed_yaml, sort_keys=False), encoding="utf-8")
print("Detection-only dataset:", DETECTION_DATASET_ROOT)
print(DATA_YAML.read_text(encoding="utf-8"))


## 2. Visual label sanity check

This cell draws the dataset annotations itself, before YOLO is trained. Confirm that boxes and class names line up with the objects.

In [ ]:
from PIL import ImageDraw

def draw_yolo_labels(image_path: Path) -> Image.Image:
    split_folder = image_path.parent.parent.name
    label_path = DATASET_ROOT / split_folder / "labels" / f"{image_path.stem}.txt"
    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)
    width, height = image.size
    if label_path.is_file():
        for raw_line in label_path.read_text(encoding="utf-8").splitlines():
            if not raw_line.strip():
                continue
            class_id, x, y, box_width, box_height, _ = parse_yolo_annotation(raw_line)
            x1 = (x - box_width / 2) * width
            y1 = (y - box_height / 2) * height
            x2 = (x + box_width / 2) * width
            y2 = (y + box_height / 2) * height
            draw.rectangle((x1, y1, x2, y2), outline="red", width=3)
            draw.text((x1 + 3, max(0, y1 - 14)), PROJECT_NAMES[class_id], fill="red")
    return image

samples = random.sample(split_images["train"], min(12, len(split_images["train"])))
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for axis, image_path in zip(axes.flat, samples):
    axis.imshow(draw_yolo_labels(image_path))
    axis.set_title(image_path.name[:35], fontsize=8)
    axis.axis("off")
plt.tight_layout()


## 3. Train YOLO26s

`SMOKE_TEST=True` runs a short pipeline check. Set it back to `False` for the real training run. The recommended full run uses 80 epochs, early stopping, AMP, and moderate augmentation because this dataset is already augmented.

In [ ]:
SMOKE_TEST = False

MODEL_NAME = "yolo26s.pt"
RUN_NAME = "illegal_tools_yolo26s_640"
EPOCHS = 2 if SMOKE_TEST else 80
FRACTION = 0.03 if SMOKE_TEST else 1.0

model = YOLO(MODEL_NAME)
model.info()

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    patience=15,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    project=str(WORK_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="auto",
    cos_lr=True,
    close_mosaic=10,
    amp=True,
    seed=SEED,
    deterministic=True,
    cache=False,
    fraction=FRACTION,
    plots=True,
    save=True,
    save_period=10,
    # Moderate augmentation; source data already contains flip/blur/noise variants.
    hsv_h=0.015,
    hsv_s=0.50,
    hsv_v=0.35,
    degrees=7.0,
    translate=0.10,
    scale=0.35,
    shear=2.0,
    perspective=0.0005,
    fliplr=0.50,
    mosaic=0.50,
    mixup=0.05,
)

RUN_DIR = WORK_ROOT / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
assert BEST_PT.is_file(), f"best.pt was not created at {BEST_PT}"
print("Best checkpoint:", BEST_PT)


### Optional: resume an interrupted Kaggle run

Kaggle sessions are temporary. If the training cell was interrupted but `last.pt` still exists in the current session, uncomment and run the cell below.

In [ ]:
# resume_model = YOLO(str(LAST_PT))
# resume_model.train(resume=True)


## 4. Evaluate both YOLO26 heads on the held-out test set

YOLO26 contains an end-to-end one-to-one head and a traditional one-to-many head. The latter can favor recall, which is useful for small/partially occluded phones. Both are measured below; predictions later use `end2end=False`.

In [ ]:
best_model = YOLO(str(BEST_PT))

metrics_e2e = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    conf=0.001,
    iou=0.70,
    plots=True,
    project=str(WORK_ROOT),
    name="test_e2e",
    end2end=True,
)

metrics_recall = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    conf=0.001,
    iou=0.70,
    plots=True,
    project=str(WORK_ROOT),
    name="test_one_to_many",
    end2end=False,
)

summary = pd.DataFrame([
    {
        "head": "end-to-end",
        "mAP50": metrics_e2e.box.map50,
        "mAP50-95": metrics_e2e.box.map,
        "precision": metrics_e2e.box.mp,
        "recall": metrics_e2e.box.mr,
    },
    {
        "head": "one-to-many",
        "mAP50": metrics_recall.box.map50,
        "mAP50-95": metrics_recall.box.map,
        "precision": metrics_recall.box.mp,
        "recall": metrics_recall.box.mr,
    },
])
display(summary)

per_class = pd.DataFrame({
    "class": PROJECT_NAMES,
    "mAP50-95_one_to_many": metrics_recall.box.maps,
})
display(per_class.sort_values("mAP50-95_one_to_many"))


In [ ]:
results_csv = RUN_DIR / "results.csv"
if results_csv.is_file():
    history = pd.read_csv(results_csv)
    display(history.tail())

plot_candidates = [
    RUN_DIR / "results.png",
    WORK_ROOT / "test_one_to_many" / "confusion_matrix_normalized.png",
    WORK_ROOT / "test_one_to_many" / "PR_curve.png",
]
existing_plots = [path for path in plot_candidates if path.is_file()]
if existing_plots:
    fig, axes = plt.subplots(1, len(existing_plots), figsize=(8 * len(existing_plots), 6))
    if len(existing_plots) == 1:
        axes = [axes]
    for axis, path in zip(axes, existing_plots):
        axis.imshow(Image.open(path))
        axis.set_title(path.name)
        axis.axis("off")
    plt.tight_layout()


## 5. Visual inference at high resolution

Inference uses `imgsz=960`, a low preview threshold, and the one-to-many head to expose difficult small objects. Production code should still apply temporal confirmation, as the current project does.

In [ ]:
preview_sources = random.sample(split_images["test"], min(16, len(split_images["test"])))
prediction_results = best_model.predict(
    source=[str(path) for path in preview_sources],
    imgsz=960,
    conf=0.15,
    iou=0.60,
    max_det=100,
    device=0,
    end2end=False,
    save=True,
    project=str(WORK_ROOT),
    name="test_predictions_960",
    exist_ok=True,
    verbose=False,
)

prediction_dir = WORK_ROOT / "test_predictions_960"
rendered = list_images(prediction_dir)
fig, axes = plt.subplots(4, 4, figsize=(18, 18))
for axis in axes.flat:
    axis.axis("off")
for axis, image_path in zip(axes.flat, rendered[:16]):
    axis.imshow(Image.open(image_path))
    axis.set_title(image_path.name[:35], fontsize=8)
plt.tight_layout()


## 6. Package and download the trained checkpoint

The final `.pt` file is copied to `/kaggle/working`, which also makes it visible in Kaggle's **Output** panel after **Save Version**.

In [ ]:
FINAL_PT = Path("/kaggle/working/yolo26s_illegal_tools_best.pt")
shutil.copy2(BEST_PT, FINAL_PT)

metadata = {
    "source_dataset": "https://www.kaggle.com/datasets/ahmedezzat02/datazeft/data",
    "base_model": MODEL_NAME,
    "class_names": PROJECT_NAMES,
    "image_size": 640,
    "recommended_inference_size_for_small_objects": 960,
    "recommended_end2end": False,
    "seed": SEED,
    "test_one_to_many": {
        "map50": float(metrics_recall.box.map50),
        "map50_95": float(metrics_recall.box.map),
        "precision": float(metrics_recall.box.mp),
        "recall": float(metrics_recall.box.mr),
    },
}
METADATA_JSON = Path("/kaggle/working/yolo26s_illegal_tools_metadata.json")
METADATA_JSON.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

ARCHIVE_BASE = "/kaggle/working/yolo26s_illegal_tools_training_run"
ARCHIVE_ZIP = Path(shutil.make_archive(ARCHIVE_BASE, "zip", root_dir=RUN_DIR))

print("Checkpoint:", FINAL_PT, FINAL_PT.stat().st_size, "bytes")
print("Metadata:", METADATA_JSON)
print("Training archive:", ARCHIVE_ZIP)
display(FileLink(str(FINAL_PT)))
display(FileLink(str(METADATA_JSON)))
display(FileLink(str(ARCHIVE_ZIP)))


## Optional ONNX export

The PyTorch checkpoint is enough for the current project. Enable this only if an ONNX deployment is needed.

In [ ]:
EXPORT_ONNX = False
if EXPORT_ONNX:
    onnx_path = best_model.export(
        format="onnx",
        imgsz=640,
        dynamic=True,
        simplify=True,
        end2end=False,
    )
    print("ONNX:", onnx_path)


## Integration checklist for this repository

1. Download `yolo26s_illegal_tools_best.pt`.
2. Copy it into the repository's `weights/` directory.
3. Point `settings.yolo_model_path` to that checkpoint.
4. Keep `book → cheat_sheet`, `Mobile_phone → smartphone`, and `smart_watch → smartwatch` as defined in this notebook.
5. If `cap`, `headset`, and `sunglasses` should generate alerts, add them to `settings.flagged_classes`.
6. For maximum small-object recall with YOLO26, call prediction/validation using `end2end=False`; keep the existing temporal `3-of-5` confirmation to suppress isolated false detections.
7. Test the new checkpoint on the project's `smartphone.mp4` and `cheatsheet.mp4` before replacing the production checkpoint.

Important limitation: this public dataset does not contain a separate `test_paper` class. It can train `cheat_sheet`/book identity, but distinguishing an authorized exam paper from a cheat sheet still depends on the project's stable `paper_id`, owner association, and paper-count policy.